# 05 · Recurrentes propios (Cisco)

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "data.py").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Ejecutar desde el repositorio DL-LAB09")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display, Image, Markdown
from src import data as D, metrics as M, backtest as B
RUN_TRAIN = False  # Cambiar a True solo para generar artefactos que falten.


Tres estrategias: LSTM de un paso con realimentación, LSTM de 24 salidas y seq2seq con calendario y teacher forcing solo en train. Se comparan dos tamaños de estado en validación. El código está en `src/models/rnn.py`; no se usan targets futuros en inferencia.

In [2]:
if RUN_TRAIN:
    import subprocess
    subprocess.run([sys.executable,"-m","src.run_cisco","--stage","rnn"],cwd=ROOT,check=True)

In [3]:
names=['LSTM_AR', 'LSTM_Multi', 'Seq2Seq']
rows=[]
for name in names:
    path=ROOT / "outputs" / "results" / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(f"Falta {path}: ejecutar la etapa correspondiente")
    rows.append(json.loads(path.read_text()))
display(pd.DataFrame(rows).set_index("modelo"))

,hardware,inference_s,familia,L,estrategia,covariables,params_total,params_entrenables,train_s,train_s_search_total,...,hidden,gpu_mem_pico_MB,epochs,max_batches_per_epoch,MAE,RMSE,WAPE,MAPE,MASE,MASE_navidad
modelo,,,,,,,,,,,,,,,,,,,,,
LSTM_AR,NVIDIA GeForce RTX 5070 Laptop GPU,8.884303,Recurrentes,168,recursiva,True,19016,19016,99.270198,235.727669,...,64,666.384766,11,1000,91.329462,800.229187,0.116859,0.166205,1.159406,1.792177
LSTM_Multi,NVIDIA GeForce RTX 5070 Laptop GPU,8.898790,Recurrentes,168,multi-salida,True,111768,111768,375.074101,437.263570,...,128,1179.259766,13,1000,67.311334,507.708643,0.086127,0.135274,0.982937,1.633981
Seq2Seq,NVIDIA GeForce RTX 5070 Laptop GPU,10.916068,Recurrentes,168,seq2seq,True,141441,141441,448.639869,651.948354,...,128,1180.891602,7,1000,70.286348,583.315710,0.089934,0.121317,0.946773,1.499650


In [4]:
for family in ["LSTM_AR","LSTM_Multi","Seq2Seq"]:
    info=json.loads((ROOT/"outputs"/"runs"/family/"selection.json").read_text())
    display(Markdown(f"### {family}: selección solo con validación"))
    display(pd.read_csv(ROOT/"outputs"/"runs"/info["nombre"]/"history.csv"))

### LSTM_AR: selección solo con validación

,epoch,train_loss,val_loss,val_MAE,val_MASE,epoch_s
0,1,0.152632,0.229408,118.511536,1.383357,4.950298
1,2,0.102269,0.228276,109.872513,1.402964,4.034245
2,3,0.095649,0.221246,106.661041,1.359059,4.036961
3,4,0.090189,0.198424,98.800766,1.280934,4.035949
4,5,0.086007,0.197184,98.706291,1.254614,4.054176
5,6,0.084211,0.188201,100.263039,1.250997,4.041522
6,7,0.081811,0.181653,87.373047,1.195208,4.043993
7,8,0.079943,0.182643,95.500420,1.214062,4.045723
8,9,0.078990,0.179957,94.681808,1.205149,4.036707
9,10,0.076751,0.174302,90.323822,1.209552,4.048550


### LSTM_Multi: selección solo con validación

,epoch,train_loss,val_loss,val_MAE,val_MASE,epoch_s
0,1,0.224105,0.171641,106.087570,1.221032,13.326897
1,2,0.156028,0.153455,94.220718,1.167952,13.341381
2,3,0.142108,0.144161,90.317574,1.131050,13.339861
3,4,0.134976,0.136762,88.368057,1.095019,13.290441
4,5,0.128856,0.131582,89.109894,1.081306,13.285667
5,6,0.125525,0.129894,87.169540,1.068440,13.286102
6,7,0.121564,0.128996,90.387154,1.054044,13.290454
7,8,0.118752,0.128082,86.943787,1.051201,13.310947
8,9,0.116926,0.123837,87.646782,1.026116,13.308204
9,10,0.115253,0.127656,88.821121,1.061736,13.293865


### Seq2Seq: selección solo con validación

,epoch,train_loss,val_loss,val_MAE,val_MASE,epoch_s
0,1,0.171658,0.195780,104.665337,1.312565,27.750928
1,2,0.112594,0.171722,97.653709,1.202353,27.758900
2,3,0.100307,0.149255,89.422821,1.122920,27.848168
3,4,0.093440,0.144209,87.804794,1.081838,27.815004
4,5,0.088070,0.137042,89.615044,1.066670,27.796392
5,6,0.084735,0.134577,85.954918,1.044956,27.781913
6,7,0.080751,0.122075,79.859627,1.008540,27.778063
7,8,0.078267,0.123239,77.445686,0.987257,27.788321
8,9,0.075953,0.124381,77.010277,0.979268,27.839307


Cada epoch usa una muestra de 1,000 batches de 512 ventanas, re-muestreada con seed por epoch (hasta 15 epochs, early stopping con validación). Stride=1 en ventanas, 150 clientes y 92 orígenes al evaluar.